Sourmash ANI pairing testEvery isolate was sequenced twice: long reads (ONT) and short reads (Illumina).The two read sets are assembled independently, and all assemblies are comparedall-vs-all with `sourmash compare --ani` (`ani_k31.csv`, k = 31).For each ONT library we ask a single question: **is its own Illumina library theclosest Illumina assembly?** If some other library is closer, the ONT andIllumina reads may have been paired incorrectly.One situation produces a "failure" that is not a mislabelling: the **resequencedisolates** (`.1` / `.2` libraries). These were sequenced twice on ONT but reuse asingle Illumina run, so the two Illumina assemblies of an isolate are effectivelyidentical and neither ONT library can prefer its own. In this dataset thelibraries named `.1`/`.2` and the strains sharing an Illumina accession areexactly the same 32 libraries, so one flag covers both — the notebook assertsthat below. (The separately resequenced `_v2` libraries do have their ownIllumina runs and are held to the test like any other library.)The notebook runs the test twice: once over all 316 libraries, and once over the294 libraries that were released as the final assembly set.

In [1]:
from pathlib import Path

import pandas as pd

TABLES = Path("../pipeline_tables")   # the stage summary tables this repo ships

ANI_MATRIX = TABLES / "ani_k31_postfix.csv.gz"   # sourmash compare --ani output
TIE = 1e-4                                       # |margin| below this is a tie, not a call


## Loading

In [2]:
def load_ani(path):
    """Read a sourmash ANI matrix into a square DataFrame labelled `SET__library`."""
    raw = pd.read_csv(path)
    labels = [c.split('/')[-1].replace('.fasta', '') for c in raw.columns]
    return pd.DataFrame(raw.values, index=labels, columns=labels)


def cross_matrix(A, rows='ONT', cols='ILL'):
    """Slice out the rows-set x cols-set block, relabelled by bare library name."""
    tag = pd.Series(A.index, index=A.index).str.split('__', n=1, expand=True)
    r, c = A.index[tag[0] == rows], A.index[tag[0] == cols]
    S = A.loc[r, c]
    S.index, S.columns = tag.loc[r, 1].values, tag.loc[c, 1].values
    return S

For every library, rank all Illumina assemblies by ANI to its ONT assembly andrecord where the self-match lands. `margin` is the ANI advantage of theself-match over the best competitor: negative means something else won.

In [3]:
def pairing_test(S, libraries=None):
    """Rank each library's own Illumina assembly among all Illumina assemblies."""
    if libraries is not None:
        keep = [l for l in libraries if l in S.index and l in S.columns]
        S = S.loc[keep, keep]

    rows = []
    for lib in S.index:
        if lib not in S.columns:
            continue
        ranked = S.loc[lib].sort_values(ascending=False)
        other = ranked.drop(lib)
        rows.append({'library': lib,
                     'ani_self': S.loc[lib, lib],
                     'rank_self': list(ranked.index).index(lib) + 1,
                     'best_other': other.index[0],
                     'ani_best_other': other.iloc[0],
                     'margin': S.loc[lib, lib] - other.iloc[0]})
    return pd.DataFrame(rows)

Sequence types come from Kleborate on the ONT assemblies; single-locus variants(`ST743-1LV`) are collapsed onto their parent ST before comparison. Strains thatshare an Illumina accession are read off the manifest.

In [4]:
ST = pd.read_csv(TABLES / "internal_qc_summary.tsv", sep='\t',
                 usecols=['sample', 'kbt_ont_ST']).set_index('sample')['kbt_ont_ST']

_man = pd.read_csv(TABLES / "sample_manifest.csv")
_dup = _man.illumina_accession[_man.illumina_accession.duplicated(keep=False)]
SHARED_ILLUMINA = set(_man.loc[_man.illumina_accession.isin(_dup), 'strain_id'])

FINAL_LIBRARIES = pd.read_csv(TABLES / "per_isolate.tsv",
                              sep='\t')['library'].tolist()

_reseq = set(_man.strain_id[_man.strain_id.str.contains(r'\.[12]$')])
assert _reseq == SHARED_ILLUMINA, 'resequencing and shared-run flags diverge'

print(f'{len(SHARED_ILLUMINA)} strains share an Illumina accession '
      f'(exactly the {len(_reseq)} libraries named .1/.2)')
print(f'{len(FINAL_LIBRARIES)} libraries in the final released set')

32 strains share an Illumina accession (exactly the 32 libraries named .1/.2)
294 libraries in the final released set


In [5]:
def base_ST(s):
    """Collapse single-locus variants: ST743-1LV -> ST743."""
    return s.astype(str).str.replace(r'-\d+LV$', '', regex=True)


def annotate(r):
    r = r.copy()
    r['ST'] = r.library.map(ST)
    r['ST_other'] = r.best_other.map(ST)
    r['same_ST'] = base_ST(r.ST) == base_ST(r.ST_other)
    r['shared_illumina'] = r.library.isin(SHARED_ILLUMINA)
    r['resequenced'] = r.library.str.contains(r'\.[12]$')

    r['verdict'] = 'self_best'
    r.loc[r.margin.abs() < TIE, 'verdict'] = 'tie'
    r.loc[r.margin < -TIE, 'verdict'] = 'other_better'
    return r

## Reporting

In [6]:
def report(r, label):
    print(f'=== {label}: {len(r)} libraries ===\n')
    print('rank of self-match')
    print(r.rank_self.value_counts().sort_index().to_string(), '\n')
    print('verdict')
    print(r.verdict.value_counts().to_string(), '\n')

    mis = r[r.rank_self > 1]
    u = unexplained(r)
    print(f'{len(mis)} libraries where another Illumina assembly ranks first:')
    print(f'  {len(mis) - len(u)} on a reused Illumina run (resequenced isolate)')
    print(f'  {len(u)} unexplained')


def unexplained(r):
    """Rank>1 libraries whose Illumina run is their own, i.e. genuine failures."""
    return r[(r.rank_self > 1) & ~r.shared_illumina]


COLS = ['library', 'best_other', 'rank_self', 'margin', 'ST', 'ST_other', 'same_ST']

## All 316 libraries

In [7]:
S = cross_matrix(load_ani(ANI_MATRIX))
r_all = annotate(pairing_test(S))
r_all.to_csv('pairing_test.tsv', sep='\t', index=False)
report(r_all, 'all libraries')

=== all libraries: 316 libraries ===

rank of self-match
rank_self
1      278
2       17
3        7
4        3
5        1
6        1
7        1
18       1
105      1
151      1
187      1
188      1
249      1
261      1
290      1 

verdict
verdict
self_best       245
tie              61
other_better     10 

38 libraries where another Illumina assembly ranks first:
  20 on a reused Illumina run (resequenced isolate)
  18 unexplained


Everything that failed the test. The largest margins all belong to `.2`libraries: those are not ties but ONT assemblies that diverge from their ownIllumina data outright (contamination or a failed run), and they were droppedduring QC rather than released.

In [8]:
r_all[r_all.rank_self > 1][COLS + ['resequenced', 'shared_illumina']] \
    .sort_values('margin')

,library,best_other,rank_self,margin,ST,ST_other,same_ST,resequenced,shared_illumina
1,INF001.2,INF007,249,-4.071624e-02,ST1449,ST1449-1LV,True,True,True
67,INF092.2,INF311,261,-2.511893e-02,NO_KLEBORATE_CALL,ST240,False,True,True
290,KSB1_8E.2,INF269,290,-2.093271e-02,NON_KPSC,ST13,False,True,True
306,KSB2_6B.2,INF019,187,-1.629057e-02,NON_KPSC,ST20,False,True,True
283,KSB1_7B.2,INF184,105,-1.490321e-02,NON_KPSC,ST3062,False,True,True
296,KSB2_10C.2,INF184,188,-1.187848e-02,NON_KPSC,ST3062,False,True,True
51,INF071.2,INF072,151,-9.086072e-03,ST133,ST133-1LV,True,True,True
88,INF117.2,INF113,18,-7.716066e-03,ST321,ST321,True,True,True
303,KSB2_5D_v2,KSB2_4D,4,-1.318516e-03,ST35,ST35,True,False,False
69,INF093.2,INF360,3,-4.346494e-04,ST323,ST323-1LV,True,True,True


The failures on libraries with their own Illumina run. All of them lose by amargin far below the typical self-vs-best-other separation, and nearly all arewithin the same ST — i.e. these are closely related strains that ANI cannotseparate, not swapped pairs.

In [9]:
u = unexplained(r_all)
print(u.margin.agg(['min', 'max', 'mean']).to_string(), '\n')
print(u.groupby('same_ST').size().to_string(), '\n')
u[COLS].sort_values('margin')

min    -1.318516e-03
max    -1.321436e-10
mean   -8.956561e-05 

same_ST
False     2
True     16 



,library,best_other,rank_self,margin,ST,ST_other,same_ST
303,KSB2_5D_v2,KSB2_4D,4,-1.318516e-03,ST35,ST35,True
181,INF261,INF255.2_v2,7,-6.392042e-05,ST29,ST743-1LV,False
217,INF318,INF309.1,3,-4.912364e-05,ST1548,ST1548,True
166,INF246,INF282,6,-4.177245e-05,ST29,ST29,True
198,INF289,INF314,3,-2.786379e-05,ST29,ST29,True
192,INF282,INF314,3,-2.728646e-05,ST29,ST29,True
130,INF193,INF212,5,-2.140425e-05,ST681,ST681,True
263,KSB1_3I,INF309.1,3,-2.037061e-05,ST1548,ST1548,True
79,INF106,INF012,2,-1.851965e-05,ST690,ST690,True
165,INF245,INF212,4,-1.536115e-05,ST681,ST681,True


For comparison, the margin among libraries that pass cleanly (rank 1, no sharedIllumina run) — the separation is three orders of magnitude larger.

In [10]:
clean = r_all[(r_all.rank_self == 1) & ~r_all.shared_illumina]
clean.margin.describe()

count    2.660000e+02
mean     4.087648e-03
std      2.717168e-03
min      4.050763e-09
25%      1.424926e-03
50%      4.834269e-03
75%      6.582453e-03
max      1.060638e-02
Name: margin, dtype: float64

The same test restricted to the released set. Only one library per resequencedisolate is released, but the sibling's Illumina assembly still leaves the pool,so the competitor set changes and ranks are recomputed from scratch rather thanfiltered out of the run above.

In [11]:
r_final = annotate(pairing_test(S, FINAL_LIBRARIES))
r_final.to_csv('pairing_test_final294.tsv', sep='\t', index=False)
report(r_final, 'final released set')

=== final released set: 294 libraries ===

rank of self-match
rank_self
1    276
2     14
4      1
5      2
6      1 

verdict
verdict
self_best    255
tie           39 

18 libraries where another Illumina assembly ranks first:
  2 on a reused Illumina run (resequenced isolate)
  16 unexplained


In [12]:
r_final[r_final.rank_self > 1][COLS + ['resequenced', 'shared_illumina']] \
    .sort_values('margin')

,library,best_other,rank_self,margin,ST,ST_other,same_ST,resequenced,shared_illumina
201,INF318,INF309.2,2,-4.912364e-05,ST1548,ST1548,True,False,False
169,INF261,INF314,6,-4.750488e-05,ST29,ST29,True,False,False
157,INF246,INF282,5,-4.177245e-05,ST29,ST29,True,False,False
185,INF289,INF314,2,-2.786379e-05,ST29,ST29,True,False,False
179,INF282,INF314,2,-2.728646e-05,ST29,ST29,True,False,False
64,INF093.1,INF295,2,-2.189117e-05,ST323,ST323,True,True,True
121,INF193,INF212,5,-2.140425e-05,ST681,ST681,True,False,False
247,KSB1_3I,INF309.2,2,-2.037061e-05,ST1548,ST1548,True,False,False
72,INF106,INF012,2,-1.851965e-05,ST690,ST690,True,False,False
156,INF245,INF212,4,-1.536115e-05,ST681,ST681,True,False,False


In [13]:
u_final = unexplained(r_final)
print(u_final.margin.agg(['min', 'max', 'mean']).to_string(), '\n')
print(u_final.groupby('same_ST').size().to_string(), '\n')
u_final[COLS].sort_values('margin')

min    -4.912364e-05
max    -1.321436e-10
mean   -1.726271e-05 

same_ST
True    16 



,library,best_other,rank_self,margin,ST,ST_other,same_ST
201,INF318,INF309.2,2,-4.912364e-05,ST1548,ST1548,True
169,INF261,INF314,6,-4.750488e-05,ST29,ST29,True
157,INF246,INF282,5,-4.177245e-05,ST29,ST29,True
185,INF289,INF314,2,-2.786379e-05,ST29,ST29,True
179,INF282,INF314,2,-2.728646e-05,ST29,ST29,True
121,INF193,INF212,5,-2.140425e-05,ST681,ST681,True
247,KSB1_3I,INF309.2,2,-2.037061e-05,ST1548,ST1548,True
72,INF106,INF012,2,-1.851965e-05,ST690,ST690,True
156,INF245,INF212,4,-1.536115e-05,ST681,ST681,True
163,INF255.1_v2,INF314,2,-2.932565e-06,ST29,ST29,True


In [14]:
clean_final = r_final[(r_final.rank_self == 1) & ~r_final.shared_illumina]
clean_final.margin.describe()

count    2.620000e+02
mean     4.160313e-03
std      2.700317e-03
min      4.050763e-09
25%      1.490633e-03
50%      4.992042e-03
75%      6.595629e-03
max      1.060638e-02
Name: margin, dtype: float64

Is the margin a win/lose split, or a relatedness split?The losses are not simply "smaller than the wins" — the two distributions overlapin the tail. What separates cleanly is how related the nearest competitor is.

In [15]:
r_final['abs_margin'] = r_final.margin.abs()
print(r_final.groupby('same_ST').abs_margin.describe()[['count', '25%', '50%', '75%']]
      .to_string(), '\n')

lose = r_final[r_final.rank_self > 1]
win = r_final[(r_final.rank_self == 1) & ~r_final.shared_illumina]
print(f'losses:  n={len(lose):3d}  median |margin| = {lose.abs_margin.median():.2e}')
print(f'wins:    n={len(win):3d}  median  margin  = {win.margin.median():.2e}')
print(f'losses with a same-ST nearest competitor: '
      f'{lose.same_ST.sum()}/{len(lose)}')
narrow = win[win.margin < lose.abs_margin.max()]
print(f'wins narrower than the worst loss:        {len(narrow)}/{len(win)}, '
      f'all same-ST: {narrow.same_ST.all()}')

         count       25%       50%       75%
same_ST                                     
False    179.0  0.005182  0.006424  0.006883
True     115.0  0.000025  0.000695  0.001530 

losses:  n= 18  median |margin| = 1.69e-05
wins:    n=262  median  margin  = 4.99e-03
losses with a same-ST nearest competitor: 18/18
wins narrower than the worst loss:        17/262, all same-ST: True


Every released library matches its own Illumina assembly at ANI >= 0.99936. The 20that do not rank first lose by a median of 1.9e-5, against a median winning marginof 4.9e-3 — and every one of them has a same-ST nearest competitor, as do all thelibraries that win by a comparably thin margin. Within a clone the sign of themargin is arbitrary, so **losing only happens where ANI has no resolving power**;it is a resolution limit, not a mispairing.The single loss whose competitor is a different ST, `INF186` (ST480) behind`INF184` (ST3062), is the known mixed-Klebsiella sample — decontaminated andreassembled in `06_asm_fixing`, after the drafts compared here.For scale, the gross failures in the full 316 set lose by 1.2e-2 to 4.1e-2 and aremostly `NON_KPSC` or failed ONT runs; none of them survives into the released set.This test cannot exclude a swap *within* an ST, only between unrelated strains.Outputs: `pairing_test.tsv` (all 316) and `pairing_test_final294.tsv` (released set).See `README.md` for the written summary.